# LeetCode 764 — Largest Plus Sign bằng Python

Link đề: https://leetcode.com/problems/largest-plus-sign/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming (quy hoạch động), Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        ...
```

Ghi chú: bài ma trận kinh điển về **"chiều dài dãy số 1 liên tiếp theo bốn hướng"**. Mỗi ô là tâm của một dấu cộng có bậc `min(trái, phải, trên, dưới)`, trong đó mỗi số là độ dài dãy 1 liên tiếp tính cả ô tâm. Notebook đi từ cách mở rộng từng tâm `O(n^3)` tới các cách `O(n^2)`: bốn bảng riêng, một bảng gộp, `bisect` tìm mìn gần nhất và chặt nhị phân + tổng tiền tố.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lưới `n x n` toàn số `1` trừ các ô mìn (`0`); tìm **bậc lớn nhất** của dấu cộng gồm các số `1` (tâm cộng bốn cánh dài `k - 1`).
2. Nắm các công cụ Python cần thiết: tạo lưới 2 chiều đúng cách, `set` các ô mìn, bộ đếm "dãy liên tiếp", `zip(range(n), reversed(range(n)))`, `bisect`, tổng tiền tố bằng `itertools.accumulate`.
3. Hiểu quan hệ cốt lõi: **bậc của dấu cộng tại một ô = `min` của bốn độ dài dãy 1 liên tiếp (trái, phải, trên, dưới), tính cả ô tâm**.
4. Đi qua 5 cách: mở rộng từng tâm (vét cạn), bốn bảng DP, một bảng gộp (lời giải chuẩn), tra khoảng cách tới mìn gần nhất bằng `bisect`, chặt nhị phân theo bậc với tổng tiền tố; đối chiếu bằng oracle độc lập.
5. Tránh các lỗi kinh điển: trả độ dài cánh thay vì bậc, chỉ xét hai hướng, quên đặt lại bộ đếm khi gặp mìn, tạo lưới bằng `[[1]*n]*n`, khởi tạo đáp án bằng 1 khi toàn mìn.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Có một lưới nhị phân `n x n` ban đầu toàn số `1`, trừ một số ô cho trong mảng `mines`: phần tử `mines[i] = [x_i, y_i]` nghĩa là `grid[x_i][y_i] == 0`.

Trả về **bậc** của dấu cộng lớn nhất gồm các số `1` trong lưới; nếu không có thì trả `0`.

**Dấu cộng trục tọa độ bậc `k`** có tâm `grid[r][c] == 1` cùng bốn cánh dài `k - 1` đi lên, xuống, trái, phải, tất cả đều là `1`. Chú ý: ngoài các cánh có thể là `0` hay `1` tùy ý.

**Ví dụ 1**

- Input: `n = 5`, `mines = [[4,2]]`
- Output: `2`
- Giải thích: dấu cộng lớn nhất chỉ có bậc 2 (một tâm với cánh dài 1 mỗi hướng).

**Ví dụ 2**

- Input: `n = 1`, `mines = [[0,0]]`
- Output: `0`
- Giải thích: không có dấu cộng nào.

**Ràng buộc (theo đề):**

- `1 <= n <= 500`
- `1 <= mines.length <= 5000`
- `0 <= x_i, y_i < n`
- Mọi cặp `(x_i, y_i)` là **duy nhất**.

Lưu ý: bậc `1` là dấu cộng chỉ gồm **một ô** `1` (không có cánh). Vậy nếu còn ít nhất một ô `1` thì đáp án `>= 1`.

In [1]:
from typing import List
from bisect import bisect_left
from itertools import accumulate
import random, time

examples = [
    (5, [[4, 2]], 2),
    (1, [[0, 0]], 0),
]
for n, mines, expected in examples:
    print(f"n={n} mines={mines} kỳ vọng={expected}")

n=5 mines=[[4, 2]] kỳ vọng=2
n=1 mines=[[0, 0]] kỳ vọng=0


## 3. Kiến thức nền

### 3.1. Tạo lưới 2 chiều đúng cách

`[[1] * n for _ in range(n)]` tạo `n` hàng **riêng biệt**. `[[1] * n] * n` tạo `n` tham chiếu tới **cùng một hàng**: sửa một ô là sửa cả cột.

In [2]:
n = 3
good = [[1] * n for _ in range(n)]
bad = [[1] * n] * n
good[0][1] = 0
bad[0][1] = 0
print("đúng:", good)
print("sai :", bad)

đúng: [[1, 0, 1], [1, 1, 1], [1, 1, 1]]
sai : [[1, 0, 1], [1, 0, 1], [1, 0, 1]]


### 3.2. Đặt mìn lên lưới và dùng `set`

Danh sách `mines` gồm cặp `[x, y]` với `x` là chỉ số hàng, `y` là cột. Chuyển thành `set` các tuple `(x, y)` để hỏi "ô này có là mìn?" trong `O(1)` (list không băm được, tuple thì được).

In [3]:
mines = [[4, 2], [0, 0]]
banned = {(x, y) for x, y in mines}
print(banned, (4, 2) in banned, (1, 1) in banned)

def build_grid(n, mines):
    grid = [[1] * n for _ in range(n)]
    for x, y in mines:
        grid[x][y] = 0
    return grid

def show_grid(grid, title=""):
    if title:
        print(title)
    for row in grid:
        print("  " + " ".join(f"{v:2}" for v in row))

show_grid(build_grid(5, [[4, 2]]), "lưới của ví dụ 1 (0 = mìn):")

{(4, 2), (0, 0)} True False
lưới của ví dụ 1 (0 = mìn):
   1  1  1  1  1
   1  1  1  1  1
   1  1  1  1  1
   1  1  1  1  1
   1  1  0  1  1


### 3.3. Dãy số 1 liên tiếp (run length) và bộ đếm có đặt lại

Với một hàng như `[1, 1, 0, 1, 1, 1]`, độ dài dãy 1 liên tiếp **kết thúc tại** mỗi vị trí (tính từ trái) là `[1, 2, 0, 1, 2, 3]`. Quy tắc: `run = run + 1` nếu ô là 1, ngược lại `run = 0` (đặt lại khi gặp 0). Tính từ phải sang trái được dãy tương tự theo hướng ngược lại.

In [4]:
def left_runs(row):
    out, run = [], 0
    for v in row:
        run = run + 1 if v else 0
        out.append(run)
    return out

row = [1, 1, 0, 1, 1, 1]
print("từ trái :", left_runs(row))
print("từ phải :", left_runs(row[::-1])[::-1])

từ trái : [1, 2, 0, 1, 2, 3]
từ phải : [2, 1, 0, 3, 2, 1]


### 3.4. `zip(range(n), reversed(range(n)))` — hai chỉ số chạy ngược chiều

Trong lời giải chuẩn, một vòng lặp duyệt đồng thời `j` tăng dần và `k` giảm dần: `(0, n-1), (1, n-2), ...`. Nhờ đó cùng một lượt ta cập nhật được "từ trái" và "từ phải".

In [5]:
print(list(zip(range(5), reversed(range(5)))))

[(0, 4), (1, 3), (2, 2), (3, 1), (4, 0)]


### 3.5. `bisect_left` và tổng tiền tố

- `bisect_left(a, x)` trên list tăng dần cho vị trí chèn `x` (số phần tử `< x`). Dùng để tìm **mìn gần nhất bên trái/phải** của một cột.
- **Tổng tiền tố** `pre[i + 1] = a[0] + ... + a[i]` giúp tính tổng đoạn `[l, r]` bằng `pre[r + 1] - pre[l]` trong `O(1)`. Với hàng toàn 0/1, tổng đoạn bằng độ dài đoạn nghĩa là **đoạn toàn số 1**.

In [6]:
a = [2, 5, 9]
print([bisect_left(a, x) for x in (0, 2, 3, 9, 10)])     # [0, 0, 1, 2, 3]

row = [1, 1, 0, 1, 1, 1]
pre = [0] + list(accumulate(row))
print("pre =", pre)
l, r = 3, 5
print(f"tổng đoạn [{l},{r}] =", pre[r + 1] - pre[l], "| độ dài", r - l + 1, "-> toàn 1?", pre[r + 1] - pre[l] == r - l + 1)
l, r = 1, 3
print(f"tổng đoạn [{l},{r}] =", pre[r + 1] - pre[l], "| độ dài", r - l + 1, "-> toàn 1?", pre[r + 1] - pre[l] == r - l + 1)

[0, 0, 1, 2, 3]
pre = [0, 1, 2, 2, 3, 4, 5]
tổng đoạn [3,5] = 3 | độ dài 3 -> toàn 1? True
tổng đoạn [1,3] = 2 | độ dài 3 -> toàn 1? False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Một dấu cộng bậc `k` tại `(r, c)` cần: `c - (k-1) .. c + (k-1)` trên hàng `r` toàn 1 (hai cánh ngang và tâm) và `r - (k-1) .. r + (k-1)` trên cột `c` toàn 1 (hai cánh dọc). Gọi:

- `L` = số ô 1 liên tiếp tính từ tâm về bên trái (tính cả tâm), `R`, `U`, `D` tương tự.

Cánh trái có độ dài `k - 1` khi `L >= k`, và tương tự mọi hướng, nên bậc lớn nhất tại tâm đó là `min(L, R, U, D)`. Đáp án bài là **max của `min(L, R, U, D)` trên mọi ô**.

Tính tay ví dụ 1 (`n = 5`, mìn ở `(4, 2)`): ô `(2, 2)` có `L = 3, R = 3, U = 3`, nhưng `D = 2` (ô `(2,2)`, `(3,2)`, rồi `(4,2)` là mìn) -> `min = 2`. Không ô nào đạt 3 vì tâm bậc 3 phải ở giữa lưới `(2, 2)` và cần `D = 3`. Đáp án `2`.

In [7]:
grid = build_grid(5, [[4, 2]])
r, c = 2, 2
def run_from(grid, r, c, dr, dc):
    n, cnt = len(grid), 0
    while 0 <= r < n and 0 <= c < n and grid[r][c]:
        cnt += 1
        r += dr
        c += dc
    return cnt

L = run_from(grid, r, c, 0, -1)
R = run_from(grid, r, c, 0, 1)
U = run_from(grid, r, c, -1, 0)
D = run_from(grid, r, c, 1, 0)
print(f"tại ({r}, {c}): L={L} R={R} U={U} D={D} -> bậc = {min(L, R, U, D)}")

tại (2, 2): L=3 R=3 U=3 D=2 -> bậc = 2


## 5. Cách 1 — Brute Force: mở rộng từng tâm

**Ý tưởng.** Với mỗi ô `(r, c)` là `1`, thử bậc `k = 1, 2, 3, ...`: dấu cộng bậc `k` còn hợp lệ nếu bốn ô cách tâm đúng `k - 1` bước theo bốn hướng (nằm trong lưới) đều là `1`. Dừng ở bậc lớn nhất hợp lệ, cập nhật đáp án.

In [8]:
def plus_brute(n: int, mines: List[List[int]]) -> int:
    grid = build_grid(n, mines)
    best = 0
    for r in range(n):
        for c in range(n):
            if not grid[r][c]:
                continue
            k = 1                                     # bậc 1: chỉ một ô tâm
            while (r - k >= 0 and r + k < n and c - k >= 0 and c + k < n
                   and grid[r - k][c] and grid[r + k][c] and grid[r][c - k] and grid[r][c + k]):
                k += 1                                # cánh dài k vẫn hợp lệ -> bậc k + 1
            best = max(best, k)
    return best

for n, mines, expected in examples:
    got = plus_brute(n, mines)
    print(n, mines, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

5 [[4, 2]] -> 2 OK
1 [[0, 0]] -> 0 OK


**Oracle độc lập.** Một oracle viết theo hướng khác: liệt kê **tập các ô của dấu cộng** rồi kiểm tra **mọi ô đều là 1 và nằm trong lưới**, thử bậc từ lớn xuống nhỏ.

In [9]:
def plus_cells(r, c, k):
    cells = {(r, c)}
    for d in range(1, k):
        cells |= {(r - d, c), (r + d, c), (r, c - d), (r, c + d)}
    return cells

def oracle(n: int, mines: List[List[int]]) -> int:
    ones = {(i, j) for i in range(n) for j in range(n)} - {(x, y) for x, y in mines}
    for k in range((n + 1) // 2, 0, -1):             # bậc tối đa có thể là (n+1)//2
        for r in range(n):
            for c in range(n):
                if plus_cells(r, c, k) <= ones:      # tập con: mọi ô là 1 (kể cả nằm trong lưới)
                    return k
    return 0

for n, mines, expected in examples:
    assert oracle(n, mines) == expected
print("oracle khớp ví dụ đề; n=3 không mìn:", oracle(3, []), "| n=5 không mìn:", oracle(5, []))

oracle khớp ví dụ đề; n=3 không mìn: 2 | n=5 không mìn: 3


**Độ phức tạp và nút thắt.** Mỗi ô thử tối đa `O(n)` bậc, tổng `O(n^3)` thời gian (với `n = 500` là hàng chục triệu bước: chậm trong Python), bộ nhớ `O(n^2)` cho lưới. Nút thắt: **các ô gần nhau lặp lại cùng một công việc** (dãy 1 liên tiếp của ô `(r, c)` và ô `(r, c+1)` chia sẻ phần lớn nội dung). Ta nên tính dãy liên tiếp **một lần** cho cả hàng/cột.

In [10]:
for size in (20, 40, 80):
    random.seed(size)
    mines = random.sample([[i, j] for i in range(size) for j in range(size)], size // 4)
    t0 = time.perf_counter()
    plus_brute(size, mines)
    print(f"n={size:3}: vét cạn mất {1000 * (time.perf_counter() - t0):8.1f} ms")

n= 20: vét cạn mất      0.3 ms
n= 40: vét cạn mất      1.6 ms
n= 80: vét cạn mất     13.1 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Tính dãy liên tiếp bằng DP một chiều.** Độ dài dãy 1 liên tiếp kết thúc tại `(i, j)` từ bên trái: `L[i][j] = L[i][j-1] + 1` nếu `grid[i][j] == 1`, ngược lại `0`. Tương tự `R` (từ phải), `U` (từ trên), `D` (từ dưới). Mỗi giá trị `O(1)`, cả bốn bảng `O(n^2)`. Đáp án `max(min(L, R, U, D))`.

Các cách tối ưu hơn:

- **Gộp bốn bảng thành một:** khởi tạo `dp = n` (giá trị "lớn"), mìn bằng `0`, rồi trong mỗi lượt cập nhật `dp[i][j] = min(dp[i][j], run)`. Giữ nguyên `O(n^2)` thời gian, bộ nhớ chỉ một bảng.
- **Thưa mìn:** vì `mines <= 5000` nhỏ so với `n^2`, dãy 1 liên tiếp của một ô đơn giản là **khoảng cách tới mìn gần nhất** (hoặc biên) theo từng hướng; lưu mìn theo hàng/cột đã sắp xếp rồi dùng `bisect`.
- **Chặt nhị phân theo bậc:** nếu có dấu cộng bậc `k` thì có bậc `k - 1`; kiểm tra "có bậc `k`?" bằng tổng tiền tố theo hàng/cột trong `O(n^2)`; tổng `O(n^2 log n)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Bốn bảng `L`, `R`, `U`, `D` rồi `min` |
| 3 | Một bảng `dp` khởi tạo `n`, cập nhật `min` trong hai lượt (một lượt hai chiều) |
| 4 | `bisect` trên mìn theo hàng/cột, khoảng cách tới mìn gần nhất |
| 5 | Chặt nhị phân bậc + tổng tiền tố hàng/cột |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bốn bảng DP

**Giải thích từng đoạn.**

- `grid` dựng từ `n` và `mines`.
- `L[i][j]`: duyệt `j` tăng; `R[i][j]`: duyệt `j` giảm; `U[i][j]`: duyệt `i` tăng; `D[i][j]`: duyệt `i` giảm.
- Với mỗi ô, bậc = `min(L, R, U, D)`; đáp án là giá trị lớn nhất.

In [11]:
def plus_four(n: int, mines: List[List[int]]) -> int:
    grid = build_grid(n, mines)
    L = [[0] * n for _ in range(n)]
    R = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                L[i][j] = (L[i][j - 1] if j else 0) + 1
                U[i][j] = (U[i - 1][j] if i else 0) + 1
    for i in range(n - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if grid[i][j]:
                R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
                D[i][j] = (D[i + 1][j] if i + 1 < n else 0) + 1
    return max(min(L[i][j], R[i][j], U[i][j], D[i][j]) for i in range(n) for j in range(n))

for n, mines, expected in examples:
    assert plus_four(n, mines) == expected
print("bốn bảng OK")

bốn bảng OK


**Code nộp LeetCode (Cách 2):**

In [12]:
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        grid = [[1] * n for _ in range(n)]
        for x, y in mines:
            grid[x][y] = 0
        L = [[0] * n for _ in range(n)]
        R = [[0] * n for _ in range(n)]
        U = [[0] * n for _ in range(n)]
        D = [[0] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                if grid[i][j]:
                    L[i][j] = (L[i][j - 1] if j else 0) + 1
                    U[i][j] = (U[i - 1][j] if i else 0) + 1
        for i in range(n - 1, -1, -1):
            for j in range(n - 1, -1, -1):
                if grid[i][j]:
                    R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
                    D[i][j] = (D[i + 1][j] if i + 1 < n else 0) + 1
        return max(min(L[i][j], R[i][j], U[i][j], D[i][j]) for i in range(n) for j in range(n))

sol_four = Solution().orderOfLargestPlusSign
print([sol_four(n, m) for n, m, _ in examples])

[2, 0]


**Độ phức tạp (Cách 2):** thời gian `O(n^2)`; bộ nhớ `O(n^2)` (năm bảng hằng số).

---

### 7.2. Cách 3 — Một bảng `dp`, cập nhật `min` ngay trong lượt quét

**Ý tưởng.** Một bảng `dp` khởi tạo mọi ô bằng `n` (đủ lớn), các ô mìn đặt `0`. Với mỗi hàng `i` và cột `i`, ta chạy **bốn bộ đếm** `left, right, up, down` và cập nhật `dp[...] = min(dp[...], bộ đếm tương ứng)`. Sau khi duyệt hết, `dp[i][j]` bằng đúng `min(L, R, U, D)`.

**Giải thích từng đoạn.**

- Ở mỗi `i`, vòng lặp `for j, k in zip(range(n), reversed(range(n)))` duyệt **hàng `i`** từ trái (`j`) và từ phải (`k`), **cột `i`** từ trên (`j`) và từ dưới (`k`).
- Bộ đếm: `left = left + 1 if dp[i][j] else 0` (đặt lại khi gặp mìn); tương tự `right, up, down`.
- Chú ý `dp[x][y]` vừa là "ô có phải mìn" (khác 0 hay 0) vừa là giá trị đang được `min`. Điều này an toàn vì `min(giá trị >= 1, run >= 1) >= 1` với ô khác mìn, nên ô khác 0 không bao giờ biến thành 0.

In [13]:
def plus_single(n: int, mines: List[List[int]]) -> int:
    dp = [[n] * n for _ in range(n)]
    for x, y in mines:
        dp[x][y] = 0
    for i in range(n):
        left = right = up = down = 0
        for j, k in zip(range(n), reversed(range(n))):
            left = left + 1 if dp[i][j] else 0
            right = right + 1 if dp[i][k] else 0
            up = up + 1 if dp[j][i] else 0
            down = down + 1 if dp[k][i] else 0
            dp[i][j] = min(dp[i][j], left)
            dp[i][k] = min(dp[i][k], right)
            dp[j][i] = min(dp[j][i], up)
            dp[k][i] = min(dp[k][i], down)
    return max(max(v) for v in dp)

for n, mines, expected in examples:
    assert plus_single(n, mines) == expected
print("một bảng OK")

một bảng OK


**Code nộp LeetCode (Cách 3):**

In [14]:
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        dp = [[n] * n for _ in range(n)]
        for x, y in mines:
            dp[x][y] = 0
        for i in range(n):
            left = right = up = down = 0
            for j, k in zip(range(n), reversed(range(n))):
                left = left + 1 if dp[i][j] else 0
                right = right + 1 if dp[i][k] else 0
                up = up + 1 if dp[j][i] else 0
                down = down + 1 if dp[k][i] else 0
                dp[i][j] = min(dp[i][j], left)
                dp[i][k] = min(dp[i][k], right)
                dp[j][i] = min(dp[j][i], up)
                dp[k][i] = min(dp[k][i], down)
        return max(max(v) for v in dp)

sol_single = Solution().orderOfLargestPlusSign
print([sol_single(n, m) for n, m, _ in examples])

[2, 0]


**Độ phức tạp (Cách 3):** thời gian `O(n^2)`, bộ nhớ `O(n^2)` (một bảng). **Đây là cách nên nộp.**

---

### 7.3. Cách 4 — `bisect` tìm mìn gần nhất theo hàng/cột

**Ý tưởng khác.** Với `mines` thưa, dãy 1 liên tiếp của một ô chính là khoảng cách tới mìn (hoặc biên lưới) gần nhất theo từng hướng. Lưu `rows[r]` = danh sách cột có mìn của hàng `r` (đã sắp xếp), `cols[c]` = danh sách hàng có mìn của cột `c`.

**Giải thích từng đoạn.**

- Với ô `(r, c)` không phải mìn: `ms = rows[r]`, `p = bisect_left(ms, c)` là số mìn nằm bên trái cột `c`. Mìn gần nhất bên trái là `ms[p - 1]` (nếu `p > 0`, ngược lại coi là `-1`); bên phải là `ms[p]` (nếu `p < len`, ngược lại coi là `n`).
- Độ dài dãy 1 tính cả tâm: `L = c - (mìn trái)`, `R = (mìn phải) - c`. Tương tự `U`, `D` với `cols[c]`.
- Bậc = `min(L, R, U, D)`.

In [15]:
def plus_bisect(n: int, mines: List[List[int]]) -> int:
    rows = [[] for _ in range(n)]
    cols = [[] for _ in range(n)]
    banned = set()
    for x, y in mines:
        rows[x].append(y)
        cols[y].append(x)
        banned.add((x, y))
    for lst in rows:
        lst.sort()
    for lst in cols:
        lst.sort()
    best = 0
    for r in range(n):
        for c in range(n):
            if (r, c) in banned:
                continue
            ms = rows[r]
            p = bisect_left(ms, c)
            left = c - (ms[p - 1] if p > 0 else -1)
            right = (ms[p] if p < len(ms) else n) - c
            ns = cols[c]
            q = bisect_left(ns, r)
            up = r - (ns[q - 1] if q > 0 else -1)
            down = (ns[q] if q < len(ns) else n) - r
            best = max(best, min(left, right, up, down))
    return best

for n, mines, expected in examples:
    assert plus_bisect(n, mines) == expected
print("bisect OK")

bisect OK


**Code nộp LeetCode (Cách 4):**

In [16]:
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        from bisect import bisect_left
        rows = [[] for _ in range(n)]
        cols = [[] for _ in range(n)]
        banned = set()
        for x, y in mines:
            rows[x].append(y)
            cols[y].append(x)
            banned.add((x, y))
        for lst in rows:
            lst.sort()
        for lst in cols:
            lst.sort()
        best = 0
        for r in range(n):
            for c in range(n):
                if (r, c) in banned:
                    continue
                ms = rows[r]
                p = bisect_left(ms, c)
                left = c - (ms[p - 1] if p > 0 else -1)
                right = (ms[p] if p < len(ms) else n) - c
                ns = cols[c]
                q = bisect_left(ns, r)
                up = r - (ns[q - 1] if q > 0 else -1)
                down = (ns[q] if q < len(ns) else n) - r
                best = max(best, min(left, right, up, down))
        return best

sol_bisect = Solution().orderOfLargestPlusSign
print([sol_bisect(n, m) for n, m, _ in examples])

[2, 0]


**Độ phức tạp (Cách 4):** thời gian `O(n^2 log m + m log m)` với `m = len(mines)` (mỗi ô 2 lần `bisect` trên danh sách nhỏ); bộ nhớ `O(n + m)` (không cần lưới!). Khi mìn rất thưa thì các danh sách rất ngắn.

---

### 7.4. Cách 5 — Chặt nhị phân theo bậc + tổng tiền tố

**Ý tưởng khác.** Tính **đơn điệu**: nếu tồn tại dấu cộng bậc `k` thì tồn tại bậc `k - 1` (chính tâm đó với cánh ngắn hơn). Do đó có thể **chặt nhị phân** bậc `k` trong `[0, (n + 1)//2]`, với hàm kiểm tra `exists(k)`: "có tâm nào mà đoạn ngang `[c-k+1, c+k-1]` trên hàng `r` toàn 1 **và** đoạn dọc `[r-k+1, r+k-1]` trên cột `c` toàn 1?"

**Giải thích từng đoạn.**

- Tổng tiền tố theo hàng `rowpre[r]` và theo cột `colpre[c]` của lưới 0/1.
- Đoạn toàn 1 khi tổng đoạn bằng độ dài đoạn `2k - 1`.
- Chỉ tâm `r, c` trong `[k - 1, n - k]` mới đủ chỗ cho bốn cánh.
- Chặt nhị phân tìm `k` lớn nhất thỏa `exists`.

In [17]:
def plus_binary(n: int, mines: List[List[int]]) -> int:
    grid = build_grid(n, mines)
    rowpre = [[0] + list(accumulate(row)) for row in grid]
    colpre = [[0] + list(accumulate(grid[r][c] for r in range(n))) for c in range(n)]

    def exists(k: int) -> bool:
        if k == 0:
            return True
        span = 2 * k - 1
        for r in range(k - 1, n - k + 1):
            for c in range(k - 1, n - k + 1):
                if (rowpre[r][c + k] - rowpre[r][c - k + 1] == span
                        and colpre[c][r + k] - colpre[c][r - k + 1] == span):
                    return True
        return False

    lo, hi = 0, (n + 1) // 2           # lo luôn thỏa; tìm k lớn nhất thỏa
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for n, mines, expected in examples:
    assert plus_binary(n, mines) == expected
print("chặt nhị phân OK")

chặt nhị phân OK


**Code nộp LeetCode (Cách 5):**

In [18]:
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        from itertools import accumulate
        grid = [[1] * n for _ in range(n)]
        for x, y in mines:
            grid[x][y] = 0
        rowpre = [[0] + list(accumulate(row)) for row in grid]
        colpre = [[0] + list(accumulate(grid[r][c] for r in range(n))) for c in range(n)]

        def exists(k: int) -> bool:
            if k == 0:
                return True
            span = 2 * k - 1
            for r in range(k - 1, n - k + 1):
                for c in range(k - 1, n - k + 1):
                    if (rowpre[r][c + k] - rowpre[r][c - k + 1] == span
                            and colpre[c][r + k] - colpre[c][r - k + 1] == span):
                        return True
            return False

        lo, hi = 0, (n + 1) // 2
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if exists(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

sol_binary = Solution().orderOfLargestPlusSign
print([sol_binary(n, m) for n, m, _ in examples])

[2, 0]


**Độ phức tạp (Cách 5):** thời gian `O(n^2 log n)` (`log n` lần gọi `exists`, mỗi lần `O(n^2)`); bộ nhớ `O(n^2)` cho tổng tiền tố.

## 8. TRACE từng bước

### 8.1. Bốn bảng và bậc cho ví dụ 1 (`n = 5`, mìn `(4, 2)`)

Hàm `trace_four` in các bảng `L, R, U, D` và bảng `min`. Quan sát: tâm `(2, 2)` có `D = 2` nên bậc 2; đáp án 2.

In [19]:
def four_tables(n, mines):
    grid = build_grid(n, mines)
    L = [[0] * n for _ in range(n)]
    R = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                L[i][j] = (L[i][j - 1] if j else 0) + 1
                U[i][j] = (U[i - 1][j] if i else 0) + 1
    for i in range(n - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if grid[i][j]:
                R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
                D[i][j] = (D[i + 1][j] if i + 1 < n else 0) + 1
    M = [[min(L[i][j], R[i][j], U[i][j], D[i][j]) for j in range(n)] for i in range(n)]
    return grid, L, R, U, D, M

def trace_four(n, mines):
    grid, L, R, U, D, M = four_tables(n, mines)
    show_grid(grid, "lưới (0 = mìn):")
    show_grid(L, "L (trái):")
    show_grid(R, "R (phải):")
    show_grid(U, "U (trên):")
    show_grid(D, "D (dưới):")
    show_grid(M, "min(L, R, U, D):")
    best = max(max(row) for row in M)
    print("đáp án =", best)
    return best

assert trace_four(5, [[4, 2]]) == 2

lưới (0 = mìn):
   1  1  1  1  1
   1  1  1  1  1
   1  1  1  1  1
   1  1  1  1  1
   1  1  0  1  1
L (trái):
   1  2  3  4  5
   1  2  3  4  5
   1  2  3  4  5
   1  2  3  4  5
   1  2  0  1  2
R (phải):
   5  4  3  2  1
   5  4  3  2  1
   5  4  3  2  1
   5  4  3  2  1
   2  1  0  2  1
U (trên):
   1  1  1  1  1
   2  2  2  2  2
   3  3  3  3  3
   4  4  4  4  4
   5  5  0  5  5
D (dưới):
   5  5  4  5  5
   4  4  3  4  4
   3  3  2  3  3
   2  2  1  2  2
   1  1  0  1  1
min(L, R, U, D):
   1  1  1  1  1
   1  2  2  2  1
   1  2  2  2  1
   1  2  1  2  1
   1  1  0  1  1
đáp án = 2


### 8.2. Cách 3 từng bước trên lưới nhỏ (`n = 3`, mìn `(1, 1)`)

Trước khi chạy, `dp` = `[[3,3,3],[3,0,3],[3,3,3]]`. Mỗi `i` quét hàng `i` và cột `i` đồng thời. In ra bốn bộ đếm và `dp` sau mỗi `(j, k)` để thấy `dp` được "mài mòn" dần bằng `min`.

In [20]:
def trace_single(n, mines):
    dp = [[n] * n for _ in range(n)]
    for x, y in mines:
        dp[x][y] = 0
    show_grid(dp, "dp ban đầu:")
    for i in range(n):
        left = right = up = down = 0
        print(f"--- i = {i} (hàng {i} và cột {i}) ---")
        for j, k in zip(range(n), reversed(range(n))):
            left = left + 1 if dp[i][j] else 0
            right = right + 1 if dp[i][k] else 0
            up = up + 1 if dp[j][i] else 0
            down = down + 1 if dp[k][i] else 0
            dp[i][j] = min(dp[i][j], left)
            dp[i][k] = min(dp[i][k], right)
            dp[j][i] = min(dp[j][i], up)
            dp[k][i] = min(dp[k][i], down)
            print(f"  j={j} k={k}: left={left} right={right} up={up} down={down}  dp={dp}")
    show_grid(dp, "dp cuối:")
    return max(max(v) for v in dp)

assert trace_single(3, [[1, 1]]) == 1

dp ban đầu:
   3  3  3
   3  0  3
   3  3  3
--- i = 0 (hàng 0 và cột 0) ---
  j=0 k=2: left=1 right=1 up=1 down=1  dp=[[1, 3, 1], [3, 0, 3], [1, 3, 3]]
  j=1 k=1: left=2 right=2 up=2 down=2  dp=[[1, 2, 1], [2, 0, 3], [1, 3, 3]]
  j=2 k=0: left=3 right=3 up=3 down=3  dp=[[1, 2, 1], [2, 0, 3], [1, 3, 3]]
--- i = 1 (hàng 1 và cột 1) ---
  j=0 k=2: left=1 right=1 up=1 down=1  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 3]]
  j=1 k=1: left=0 right=0 up=0 down=0  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 3]]
  j=2 k=0: left=1 right=1 up=1 down=1  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 3]]
--- i = 2 (hàng 2 và cột 2) ---
  j=0 k=2: left=1 right=1 up=1 down=1  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 1]]
  j=1 k=1: left=2 right=2 up=2 down=2  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 1]]
  j=2 k=0: left=3 right=3 up=3 down=3  dp=[[1, 1, 1], [1, 0, 1], [1, 1, 1]]
dp cuối:
   1  1  1
   1  0  1
   1  1  1


### 8.3. Cách 4 (bisect) cho ô `(2, 2)` của ví dụ 1

`cols[2] = [4]` (mìn ở hàng 4 của cột 2). Với `r = 2`: `bisect_left([4], 2) = 0` nên không có mìn phía trên (coi là `-1`): `up = 2 - (-1) = 3`; mìn phía dưới là hàng 4: `down = 4 - 2 = 2`. Hàng 2 không có mìn: `left = 2 - (-1) = 3`, `right = 5 - 2 = 3`. Bậc = `min(3, 3, 3, 2) = 2`.

In [21]:
def trace_bisect_cell(n, mines, r, c):
    rows = [sorted(y for x, y in mines if x == i) for i in range(n)]
    cols = [sorted(x for x, y in mines if y == j) for j in range(n)]
    ms, ns = rows[r], cols[c]
    p, q = bisect_left(ms, c), bisect_left(ns, r)
    print(f"hàng {r}: mìn tại cột {ms}; p = bisect_left = {p}")
    print(f"cột  {c}: mìn tại hàng {ns}; q = bisect_left = {q}")
    left = c - (ms[p - 1] if p > 0 else -1)
    right = (ms[p] if p < len(ms) else n) - c
    up = r - (ns[q - 1] if q > 0 else -1)
    down = (ns[q] if q < len(ns) else n) - r
    print(f"left={left} right={right} up={up} down={down} -> bậc = {min(left, right, up, down)}")
    return min(left, right, up, down)

assert trace_bisect_cell(5, [[4, 2]], 2, 2) == 2
assert trace_bisect_cell(5, [[4, 2]], 4, 0) == 1

hàng 2: mìn tại cột []; p = bisect_left = 0
cột  2: mìn tại hàng [4]; q = bisect_left = 0
left=3 right=3 up=3 down=2 -> bậc = 2
hàng 4: mìn tại cột [2]; p = bisect_left = 0
cột  0: mìn tại hàng []; q = bisect_left = 0
left=1 right=2 up=5 down=1 -> bậc = 1


### 8.4. Cách 5: chặt nhị phân từng bước

Với `n = 5`, mìn `(4, 2)`: khoảng ban đầu `[0, 3]`. Thử `mid = 2` (có), rồi `mid = 3` (không). Kết quả 2.

In [22]:
def trace_binary(n, mines):
    grid = build_grid(n, mines)
    rowpre = [[0] + list(accumulate(row)) for row in grid]
    colpre = [[0] + list(accumulate(grid[r][c] for r in range(n))) for c in range(n)]

    def exists(k):
        if k == 0:
            return True, None
        span = 2 * k - 1
        for r in range(k - 1, n - k + 1):
            for c in range(k - 1, n - k + 1):
                if (rowpre[r][c + k] - rowpre[r][c - k + 1] == span
                        and colpre[c][r + k] - colpre[c][r - k + 1] == span):
                    return True, (r, c)
        return False, None

    lo, hi = 0, (n + 1) // 2
    print(f"n={n}: khoảng bậc [lo, hi] = [{lo}, {hi}]")
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok, where = exists(mid)
        print(f"  thử bậc {mid}: {'có, tâm ' + str(where) if ok else 'không có'}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
        print(f"  -> [lo, hi] = [{lo}, {hi}]")
    return lo

assert trace_binary(5, [[4, 2]]) == 2

n=5: khoảng bậc [lo, hi] = [0, 3]
  thử bậc 2: có, tâm (1, 1)
  -> [lo, hi] = [2, 3]
  thử bậc 3: không có
  -> [lo, hi] = [2, 2]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (một bảng `dp`): `O(n^2)` thời gian, một bảng bộ nhớ, lời giải chuẩn.

```python
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        dp = [[n] * n for _ in range(n)]
        for x, y in mines:
            dp[x][y] = 0
        for i in range(n):
            left = right = up = down = 0
            for j, k in zip(range(n), reversed(range(n))):
                left = left + 1 if dp[i][j] else 0
                right = right + 1 if dp[i][k] else 0
                up = up + 1 if dp[j][i] else 0
                down = down + 1 if dp[k][i] else 0
                dp[i][j] = min(dp[i][j], left)
                dp[i][k] = min(dp[i][k], right)
                dp[j][i] = min(dp[j][i], up)
                dp[k][i] = min(dp[k][i], down)
        return max(max(v) for v in dp)
```

In [23]:
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        dp = [[n] * n for _ in range(n)]
        for x, y in mines:
            dp[x][y] = 0
        for i in range(n):
            left = right = up = down = 0
            for j, k in zip(range(n), reversed(range(n))):
                left = left + 1 if dp[i][j] else 0
                right = right + 1 if dp[i][k] else 0
                up = up + 1 if dp[j][i] else 0
                down = down + 1 if dp[k][i] else 0
                dp[i][j] = min(dp[i][j], left)
                dp[i][k] = min(dp[i][k], right)
                dp[j][i] = min(dp[j][i], up)
                dp[k][i] = min(dp[k][i], down)
        return max(max(v) for v in dp)

sol = Solution()
for n, mines, expected in examples:
    got = sol.orderOfLargestPlusSign(n, mines)
    print(n, mines, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

5 [[4, 2]] -> 2 OK
1 [[0, 0]] -> 0 OK


## 10. Vì sao đúng

**(a) Bậc tại một tâm.** Dấu cộng bậc `k` tại `(r, c)` hợp lệ ⇔ `c - k + 1 .. c + k - 1` nằm trong lưới và toàn 1 trên hàng `r`, và `r - k + 1 .. r + k - 1` trên cột `c` cũng vậy. Điều này tương đương `L, R, U, D >= k` với `L` = số ô 1 liên tiếp từ tâm sang trái (tính cả tâm), vì `L >= k` nghĩa là `k - 1` ô bên trái tâm đều là 1 và không ra ngoài lưới. Bậc lớn nhất tại tâm là `min(L, R, U, D)`, đáp án là max trên các tâm (tâm là mìn có `L = 0` nên bậc 0).

**(b) DP dãy liên tiếp.** `L[i][j] = L[i][j-1] + 1` nếu ô là 1: dãy 1 kết thúc tại `(i, j)` kéo dài dãy kết thúc tại `(i, j-1)` thêm 1; nếu ô là mìn thì dãy bị đứt, `L = 0`. Bốn hướng đối xứng.

**(c) Cách 3 (một bảng).** Sau khi quét xong hàng `i` và cột `i` ở lượt `i`, mỗi ô `(i, j)` đã được `min` với `left` (hướng trái của ô), `right`, còn ô `(j, i)` với `up`, `down`. Mỗi ô `(a, b)` được quét theo hàng ở lượt `i = a` (cho `left, right`) và theo cột ở lượt `i = b` (cho `up, down`) **đúng một lần mỗi hướng**, nên cuối cùng `dp[a][b] = min(n, L, R, U, D) = min(L, R, U, D)` vì mọi run `<= n`. Mìn là 0 từ đầu và `min` không làm 0 trở thành dương, còn ô khác mìn luôn dương nên cờ `dp[...] != 0` đúng nghĩa "không phải mìn".

**(d) Cách 4.** Với ô `(r, c)` không phải mìn, ô 1 liên tiếp bên trái chỉ bị chặn bởi mìn gần nhất bên trái (hoặc biên): `L = c - vị_trí_mìn_trái` (với biên coi mìn ở `-1`), tương tự bên phải với biên ở `n`. `bisect_left(ms, c)` = số mìn `< c` (và `c` không thuộc `ms` vì ô không phải mìn) nên `ms[p-1]` là mìn trái gần nhất, `ms[p]` là mìn phải gần nhất.

**(e) Cách 5.** Đơn điệu: dấu cộng bậc `k` tại tâm `(r, c)` chứa dấu cộng bậc `k - 1` cùng tâm. Do đó tập `k` thỏa `exists(k)` là một đoạn đầu `[0, K]`, chặt nhị phân đúng. Kiểm tra bằng tổng tiền tố đúng theo định nghĩa (a).

**Kiểm chứng bằng máy:** so cả 5 cách với oracle (liệt kê ô của dấu cộng) trên **mọi** lưới `n <= 3` (mọi tập mìn có thể) và nhiều lưới ngẫu nhiên có seed.

In [24]:
solutions = {
    "brute": plus_brute,
    "four": plus_four,
    "single": plus_single,
    "bisect": plus_bisect,
    "binary": plus_binary,
    "sol_four": sol_four,
    "sol_single": sol_single,
    "sol_bisect": sol_bisect,
    "sol_binary": sol_binary,
    "final": Solution().orderOfLargestPlusSign,
}

checked = 0
for n in (1, 2, 3, 4):
    cells = [[i, j] for i in range(n) for j in range(n)]
    for mask in range(1 << (n * n)):                 # mọi tập mìn (kể cả rỗng) trên lưới n x n
        mines = [cells[t] for t in range(n * n) if mask >> t & 1]
        ref = oracle(n, mines)
        for name, fn in solutions.items():
            assert fn(n, mines) == ref, (name, n, mines, fn(n, mines), ref)
        checked += 1
print("mọi lưới n = 1..4 (mọi tập mìn):", checked, "lưới x", len(solutions), "cài đặt khớp oracle")

mọi lưới n = 1..4 (mọi tập mìn): 66066 lưới x 10 cài đặt khớp oracle


## 11. Time / Space Complexity

Gọi `n` là kích thước lưới (`<= 500`), `m = len(mines)` (`<= 5000`).

| Cách | Time | Space |
|---|---|---|
| 1. Mở rộng từng tâm | `O(n^3)` | `O(n^2)` |
| 2. Bốn bảng | `O(n^2)` | `O(n^2)` (5 bảng) |
| 3. Một bảng `dp` | `O(n^2)` | `O(n^2)` (1 bảng) |
| 4. `bisect` theo mìn | `O(n^2 log m)` | `O(n + m)` |
| 5. Chặt nhị phân + tổng tiền tố | `O(n^2 log n)` | `O(n^2)` |

Với `n = 500`: `n^2 = 250.000` ô, chạy vài trăm mili-giây tới vài giây trong Python. Vét cạn `O(n^3)` với mìn thưa là rất chậm. Hãy đo.

In [25]:
random.seed(764)
n = 500
all_cells = [[i, j] for i in range(n) for j in range(n)]
mines_big = random.sample(all_cells, 5000)
ref = None
for name in ["four", "single", "bisect", "binary"]:
    t0 = time.perf_counter()
    res = solutions[name](n, mines_big)
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:7} n=500, 5000 mìn: đáp án {res}, {dt:9.1f} ms")

n2 = 120
mines_mid = random.sample([[i, j] for i in range(n2) for j in range(n2)], 150)
t0 = time.perf_counter()
r_brute = plus_brute(n2, mines_mid)
print(f"brute   n=120, 150 mìn: đáp án {r_brute}, {1000 * (time.perf_counter() - t0):9.1f} ms")
assert r_brute == plus_single(n2, mines_mid)

four    n=500, 5000 mìn: đáp án 133,     105.5 ms


single  n=500, 5000 mìn: đáp án 133,      85.4 ms
bisect  n=500, 5000 mìn: đáp án 133,     104.9 ms


binary  n=500, 5000 mìn: đáp án 133,      47.0 ms
brute   n=120, 150 mìn: đáp án 57,      29.1 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case (có kỳ vọng viết tay), phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle trên lưới nhỏ ngẫu nhiên và với nhau trên lưới lớn ngẫu nhiên có seed cố định (đúng ràng buộc: mìn duy nhất, `1 <= m <= 5000`).

In [26]:
# 12.1 ví dụ đề
for n, mines, expected in examples:
    for name, fn in solutions.items():
        assert fn(n, mines) == expected, (name, n, mines)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay)
def diag(n):
    return [[i, i] for i in range(n)]

edge = [
    (1, [[0, 0]], 0),                        # toàn mìn
    (1, [], 1),                              # không mìn (ngoài ràng buộc của đề nhưng hợp lệ): một ô
    (2, [], 1),                              # lưới 2x2 không đủ chỗ cho cánh
    (2, [[0, 0], [1, 1]], 1),
    (2, [[0, 0], [0, 1], [1, 0], [1, 1]], 0),
    (3, [], 2),
    (3, [[1, 1]], 1),                        # mìn ngay giữa
    (3, [[0, 0]], 2),                        # mìn ở góc không ảnh hưởng tâm giữa
    (3, [[0, 1]], 1),                        # mìn ở cánh trên của tâm giữa
    (4, [], 2),
    (5, [], 3),
    (5, [[2, 2]], 2),
    (5, [[0, 0], [0, 4], [4, 0], [4, 4]], 3),
    (5, diag(5), 2),                         # đường chéo toàn mìn: tâm (1,3) vẫn đủ bậc 2
    (6, [], 3),
    (7, [], 4),
    (500, [[0, 0]], 250),                    # lưới lớn nhất, gần như trống: tâm (249,249) vẫn đủ 250 bậc
]
for n, mines, expected in edge:
    if n <= 7:
        assert oracle(n, mines) == expected, ("oracle khác kỳ vọng", n, mines, oracle(n, mines))
    for name, fn in solutions.items():
        if name == "brute" and n > 100:
            continue
        if name in ("binary", "sol_binary") and n > 100:
            continue
        assert fn(n, mines) == expected, (name, n, mines, fn(n, mines), expected)
print("12.2 edge case: OK", len(edge), "ca")

# 12.3 phản ví dụ: các cách sai
def wrong_arm_length(n, mines):      # trả độ dài cánh (bậc - 1)
    return plus_single(n, mines) - 1

def wrong_two_dirs(n, mines):        # chỉ xét trái và trên
    grid = build_grid(n, mines)
    L = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                L[i][j] = (L[i][j - 1] if j else 0) + 1
                U[i][j] = (U[i - 1][j] if i else 0) + 1
    return max(min(L[i][j], U[i][j]) for i in range(n) for j in range(n))

def wrong_no_reset(n, mines):        # bộ đếm không đặt lại về 0 khi gặp mìn (đếm tổng số 1)
    grid = build_grid(n, mines)
    best = 0
    L = [[0] * n for _ in range(n)]
    R = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        run = 0
        for j in range(n):
            run += grid[i][j]
            L[i][j] = run
        run = 0
        for j in range(n - 1, -1, -1):
            run += grid[i][j]
            R[i][j] = run
    for j in range(n):
        run = 0
        for i in range(n):
            run += grid[i][j]
            U[i][j] = run
        run = 0
        for i in range(n - 1, -1, -1):
            run += grid[i][j]
            D[i][j] = run
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                best = max(best, min(L[i][j], R[i][j], U[i][j], D[i][j]))
    return best

def wrong_alias(n, mines):           # lưới tạo bằng [[1]*n]*n
    grid = [[1] * n] * n
    for x, y in mines:
        grid[x][y] = 0
    best = 0
    for r in range(n):
        for c in range(n):
            if not grid[r][c]:
                continue
            k = 1
            while (r - k >= 0 and r + k < n and c - k >= 0 and c + k < n
                   and grid[r - k][c] and grid[r + k][c] and grid[r][c - k] and grid[r][c + k]):
                k += 1
            best = max(best, k)
    return best

def wrong_init_one(n, mines):        # khởi tạo đáp án = 1 dù có thể toàn mìn
    return max(1, plus_single(n, mines))

cases = [
    ("trả độ dài cánh", wrong_arm_length, 5, [[4, 2]]),
    ("chỉ hai hướng", wrong_two_dirs, 5, [[4, 2]]),
    ("không đặt lại bộ đếm", wrong_no_reset, 5, [[1, 1], [1, 3], [3, 2]]),
    ("lưới [[1]*n]*n", wrong_alias, 3, [[0, 0]]),
    ("đáp án khởi tạo 1", wrong_init_one, 1, [[0, 0]]),
]
for name, fn, n, mines in cases:
    truth = oracle(n, mines)
    print(f"{name:22} n={n} mines={mines}: sai = {fn(n, mines)}, đúng = {truth}")
    assert fn(n, mines) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 ngẫu nhiên nhỏ so với oracle
rng = random.Random(2764)
for _ in range(300):
    n = rng.randint(1, 8)
    cells = [[i, j] for i in range(n) for j in range(n)]
    m = rng.randint(0, min(len(cells), rng.choice([2, 5, 10, 64])))
    mines = rng.sample(cells, m)
    ref = oracle(n, mines)
    for name, fn in solutions.items():
        assert fn(n, mines) == ref, (name, n, mines)
print("12.4 ngẫu nhiên nhỏ so với oracle: OK")

# 12.5 ngẫu nhiên lớn: các cách O(n^2) đối chiếu nhau (đúng ràng buộc n <= 500, 1 <= m <= 5000, mìn duy nhất)
rng = random.Random(77)
fast = ["four", "single", "bisect", "binary", "final"]
for _ in range(12):
    n = rng.randint(30, 200)
    m = rng.randint(1, min(5000, n * n))
    mines = rng.sample([[i, j] for i in range(n) for j in range(n)], m)
    res = {name: solutions[name](n, mines) for name in fast}
    assert len(set(res.values())) == 1, (n, m, res)
    assert 0 <= res["final"] <= (n + 1) // 2
print("12.5 ngẫu nhiên lớn (n tới 200): OK")

# 12.6 lưới lớn nhất n = 500 với 5000 mìn và mìn xếp thành đường
n = 500
line = [[i, 250] for i in range(500)] + [[250, j] for j in range(500) if j != 250]
line = line[:5000]
res = {name: solutions[name](n, line) for name in ("single", "bisect")}
assert res["single"] == res["bisect"]
print("12.6 n = 500, mìn xếp thành chữ thập:", res["single"])

12.1 ví dụ đề: OK


12.2 edge case: OK 17 ca
trả độ dài cánh        n=5 mines=[[4, 2]]: sai = 1, đúng = 2
chỉ hai hướng          n=5 mines=[[4, 2]]: sai = 4, đúng = 2
không đặt lại bộ đếm   n=5 mines=[[1, 1], [1, 3], [3, 2]]: sai = 2, đúng = 1
lưới [[1]*n]*n         n=3 mines=[[0, 0]]: sai = 1, đúng = 2
đáp án khởi tạo 1      n=1 mines=[[0, 0]]: sai = 1, đúng = 0
12.3 phản ví dụ: OK (các cách sai đều bị lộ)
12.4 ngẫu nhiên nhỏ so với oracle: OK


12.5 ngẫu nhiên lớn (n tới 200): OK
12.6 n = 500, mìn xếp thành chữ thập: 125


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả độ dài cánh thay vì bậc

Bậc `k` có cánh dài `k - 1`. Nếu đếm số ô **bên cạnh** tâm (không tính tâm) thì thiếu 1.

In [27]:
n, mines = 5, [[4, 2]]
print("đúng (bậc):", plus_single(n, mines), "| sai (độ dài cánh):", wrong_arm_length(n, mines))
assert plus_single(n, mines) == 2 and wrong_arm_length(n, mines) == 1

đúng (bậc): 2 | sai (độ dài cánh): 1


Vì sao sai: đề hỏi **bậc** (order) = số ô từ tâm đến đầu cánh tính cả tâm. **Đúng:** run length **tính cả ô tâm**, như `left = left + 1 if ... else 0` bắt đầu từ ô tâm.

### Lỗi 2 — Chỉ xét hai hướng

Dấu cộng cần **bốn** cánh. Chỉ xét trái + trên cho kết quả quá lạc quan.

In [28]:
print("đúng:", plus_single(5, [[4, 2]]), "| chỉ trái+trên:", wrong_two_dirs(5, [[4, 2]]))
assert wrong_two_dirs(5, [[4, 2]]) == 4 and plus_single(5, [[4, 2]]) == 2

đúng: 2 | chỉ trái+trên: 4


Vì sao sai: ô `(3, 3)` có trái và trên dài 4 (bậc 4 nếu chỉ xét hai hướng) nhưng chỉ có cánh phải/dưới dài 2. **Đúng:** lấy `min` của **cả bốn** hướng.

### Lỗi 3 — Không đặt lại bộ đếm khi gặp mìn

`run += grid[i][j]` đếm tổng số 1 từ đầu, chứ không phải dãy **liên tiếp**.

In [29]:
n, mines = 5, [[1, 1], [1, 3], [3, 2]]
print("đúng:", plus_single(n, mines), "| không đặt lại:", wrong_no_reset(n, mines), "| oracle:", oracle(n, mines))
assert wrong_no_reset(n, mines) != plus_single(n, mines) == oracle(n, mines)

đúng: 1 | không đặt lại: 2 | oracle: 1


Vì sao sai: một mìn chia hàng thành hai đoạn; cánh không được "bắc cầu" qua mìn. **Đúng:** `run = run + 1 if ô là 1 else 0`.

### Lỗi 4 — Tạo lưới bằng `[[1] * n] * n`

Các hàng là **cùng một list**. Đặt một mìn là đặt cả cột.

In [30]:
n, mines = 3, [[0, 0]]
print("đúng:", plus_single(n, mines), "| lưới bị alias:", wrong_alias(n, mines))
assert plus_single(n, mines) == 2 and wrong_alias(n, mines) == 1

đúng: 2 | lưới bị alias: 1


Vì sao sai: `grid[0][0] = 0` làm cả cột 0 thành 0, biến lưới 3x3 thành lưới có 3 mìn (tâm giữa mất cánh trái). **Đúng:** `[[1] * n for _ in range(n)]`.

### Lỗi 5 — Khởi tạo đáp án bằng 1

Nếu mọi ô đều là mìn (ví dụ `n = 1`, `mines = [[0,0]]`) đáp án phải là `0`.

In [31]:
print("đúng:", plus_single(1, [[0, 0]]), "| khởi tạo 1:", wrong_init_one(1, [[0, 0]]))
assert plus_single(1, [[0, 0]]) == 0 and wrong_init_one(1, [[0, 0]]) == 1

đúng: 0 | khởi tạo 1: 1


Vì sao sai: bậc `1` đòi hỏi tâm là `1`. **Đúng:** khởi tạo `best = 0` và để `max` quyết định.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "dấu cộng/hình chữ thập/viền", "liên tiếp theo bốn hướng" -> bốn bảng run-length rồi `min`.
- **Gộp nhiều bảng:** khi chỉ cần `min` của các đại lượng cùng ô, dùng một bảng khởi tạo "vô cùng" và `dp = min(dp, run)` trong từng hướng.
- **Mìn thưa:** `m << n^2` -> khoảng cách tới mìn gần nhất bằng `bisect`, không cần lưới đầy đủ.
- **Đơn điệu theo bậc** -> chặt nhị phân + kiểm tra bằng tổng tiền tố.
- Cẩn thận biên: biên lưới hành xử như một "mìn" ở chỉ số `-1` hoặc `n`.
- Tâm cánh phải đủ chỗ: `k - 1 <= r, c <= n - k`; bậc tối đa `(n + 1) // 2`.
- Test nên có: `n = 1`, lưới không mìn, mìn giữa lưới, mìn ở đúng một cánh, đường chéo mìn.

## 15. Biến thể

1. **Ma trận bậc** `orders(n, mines)`: giá trị `min(L, R, U, D)` ở mọi ô.
2. **Đếm số tâm có bậc `>= k`** (bài tập 5).
3. **Largest 1-Bordered Square (1139):** cạnh một hình vuông có viền toàn 1; dùng run-length phải/xuống.
4. **Maximal Square (221), Maximal Rectangle (85)**, **Count Square Submatrices (1277)**: cùng họ DP trên ma trận.
5. **Dấu `X`** (đường chéo) thay vì dấu cộng: tương tự với bốn hướng chéo.

Dưới đây là code cho biến thể 1 và 3, kiểm bằng oracle.

In [32]:
def orders(n: int, mines: List[List[int]]) -> List[List[int]]:
    return four_tables(n, mines)[5]

show_grid(orders(5, [[4, 2]]), "ma trận bậc của ví dụ 1:")
assert max(max(r) for r in orders(5, [[4, 2]])) == 2

def largest_1_bordered_square(grid: List[List[int]]) -> int:
    """Trả diện tích hình vuông lớn nhất có viền toàn 1."""
    m, n = len(grid), len(grid[0])
    R = [[0] * n for _ in range(m)]       # số ô 1 liên tiếp sang phải kể từ (i, j)
    D = [[0] * n for _ in range(m)]       # số ô 1 liên tiếp xuống dưới
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if grid[i][j]:
                R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
                D[i][j] = (D[i + 1][j] if i + 1 < m else 0) + 1
    best = 0
    for i in range(m):
        for j in range(n):
            for k in range(min(R[i][j], D[i][j]), 0, -1):
                if R[i + k - 1][j] >= k and D[i][j + k - 1] >= k:
                    best = max(best, k * k)
                    break
    return best

def border_oracle(grid):
    m, n = len(grid), len(grid[0])
    best = 0
    for i in range(m):
        for j in range(n):
            for k in range(1, min(m - i, n - j) + 1):
                cells = [(i, j + t) for t in range(k)] + [(i + k - 1, j + t) for t in range(k)] \
                      + [(i + t, j) for t in range(k)] + [(i + t, j + k - 1) for t in range(k)]
                if all(grid[a][b] for a, b in cells):
                    best = max(best, k * k)
    return best

assert largest_1_bordered_square([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 9
assert largest_1_bordered_square([[1, 1, 0, 0]]) == 1
rng = random.Random(1139)
for _ in range(300):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.choice([0, 1, 1]) for _ in range(n)] for _ in range(m)]
    assert largest_1_bordered_square(g) == border_oracle(g), g
print("Biến thể: OK")

ma trận bậc của ví dụ 1:
   1  1  1  1  1
   1  2  2  2  1
   1  2  2  2  1
   1  2  1  2  1
   1  1  0  1  1
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mở rộng từng tâm | Thử bậc tăng dần ở mỗi ô | `O(n^3)` | `O(n^2)` | Hiểu đề; lưới nhỏ |
| 2. Bốn bảng | `L, R, U, D` rồi `min` | `O(n^2)` | `O(n^2)` (5 bảng) | Rõ ràng nhất, dễ gỡ lỗi |
| 3. Một bảng `dp` | `min` trực tiếp trong lượt quét | `O(n^2)` | `O(n^2)` (1 bảng) | **Cách nộp chính** |
| 4. `bisect` theo mìn | Khoảng cách tới mìn gần nhất | `O(n^2 log m)` | `O(n + m)` | Mìn rất thưa, lưới rất lớn |
| 5. Chặt nhị phân | Tính đơn điệu theo bậc | `O(n^2 log n)` | `O(n^2)` | Khi đã quen kiểu "chặt nhị phân đáp án" |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(left_runs([1, 1, 0, 1, 1, 1]))
print(plus_brute(3, [[1, 1]]))
print(plus_brute(3, []))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 3`, `mines = []`: điền tay ma trận bậc `min(L, R, U, D)` ở mọi ô (list 3 list) và đáp án.

In [34]:
# Bài 2: điền kết quả tính tay
hand_orders = None
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm bốn bảng (đã cho sẵn `L` và `R`).

In [35]:
def exercise_3_four(n: int, mines: List[List[int]]) -> int:
    grid = [[1] * n for _ in range(n)]
    for x, y in mines:
        grid[x][y] = 0
    L = [[0] * n for _ in range(n)]
    R = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                L[i][j] = (L[i][j - 1] if j else 0) + 1
        for j in range(n - 1, -1, -1):
            if grid[i][j]:
                R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
    # TODO 1: tính U[i][j] (số ô 1 liên tiếp từ trên xuống tới (i, j), tính cả ô đó)
    # TODO 2: tính D[i][j] (từ dưới lên)
    # TODO 3: trả về max của min(L, R, U, D) trên mọi ô
    return None


# assert exercise_3_four(5, [[4, 2]]) == 2
# assert exercise_3_four(1, [[0, 0]]) == 0
# assert exercise_3_four(3, []) == 2

### Bài 4 — Tự viết lại (vừa-khó)

Viết hàm `exercise_4_exists(n, mines, k)` trả `True` nếu tồn tại dấu cộng **bậc `k`** (`k >= 1`) trong lưới; dùng cách mở rộng trực tiếp (không cần tối ưu).

In [36]:
def exercise_4_exists(n: int, mines: List[List[int]], k: int) -> bool:
    # TODO
    pass


# assert exercise_4_exists(5, [[4, 2]], 2) is True
# assert exercise_4_exists(5, [[4, 2]], 3) is False
# assert exercise_4_exists(1, [[0, 0]], 1) is False

### Bài 5 — Mini challenge: đếm số tâm có bậc >= k (khó)

Cho `n`, `mines` và `k >= 1`, đếm có bao nhiêu **ô** là tâm của một dấu cộng bậc **ít nhất** `k`.

```python
exercise_5_count_centers(5, [], 3)          -> 1     (chỉ tâm (2, 2))
exercise_5_count_centers(5, [], 2)          -> 9     (các ô trong khối 3x3 giữa)
exercise_5_count_centers(5, [[4, 2]], 2)    -> 8
exercise_5_count_centers(3, [[1, 1]], 1)    -> 8
```
Gợi ý: dùng bảng `min(L, R, U, D)` rồi đếm ô `>= k`.

In [37]:
def exercise_5_count_centers(n: int, mines: List[List[int]], k: int) -> int:
    # TODO
    pass


# assert exercise_5_count_centers(5, [], 3) == 1
# assert exercise_5_count_centers(5, [], 2) == 9
# assert exercise_5_count_centers(5, [[4, 2]], 2) == 8
# assert exercise_5_count_centers(3, [[1, 1]], 1) == 8

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `left_runs` đặt lại về 0 khi gặp 0.
- Mức 2: lưới 3x3 có mìn ở tâm: mọi ô còn lại chỉ đủ bậc 1.
- Mức 3: `[1, 2, 0, 1, 2, 3]`, `1`, `2`.

**Bài 2**
- Mức 1: ô góc có `L = 1` hoặc `U = 1`... nên bậc 1.
- Mức 2: ô giữa `(1, 1)` có `L = R = U = D = 2`.
- Mức 3: `[[1,1,1],[1,2,1],[1,1,1]]`, đáp án `2`.

**Bài 3**
- Mức 1: `U[i][j] = (U[i - 1][j] if i else 0) + 1` khi `grid[i][j]`.
- Mức 2: `D` duyệt `i` giảm: `(D[i + 1][j] if i + 1 < n else 0) + 1`.
- Mức 3: `return max(min(L[i][j], R[i][j], U[i][j], D[i][j]) for i in range(n) for j in range(n))`.

**Bài 4**
- Mức 1: duyệt mọi tâm `(r, c)`; cần `r - (k-1) >= 0` và `r + (k-1) < n` (tương tự cột).
- Mức 2: kiểm tra tâm và `4 * (k - 1)` ô cánh đều không phải mìn.
- Mức 3: tập `banned = {(x, y)}`; `all((r + d, c) not in banned and (r - d, c) not in banned and (r, c + d) not in banned and (r, c - d) not in banned for d in range(0, k))`.

**Bài 5**
- Mức 1: tái sử dụng `four_tables(n, mines)[5]` hoặc tự viết bảng `min`.
- Mức 2: bậc `>= k` nghĩa là `min(L, R, U, D) >= k`.
- Mức 3: `sum(1 for row in M for v in row if v >= k)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(left_runs([1, 1, 0, 1, 1, 1]))
print(plus_brute(3, [[1, 1]]))
print(plus_brute(3, []))

assert left_runs([1, 1, 0, 1, 1, 1]) == [1, 2, 0, 1, 2, 3]
assert plus_brute(3, [[1, 1]]) == 1
assert plus_brute(3, []) == 2

[1, 2, 0, 1, 2, 3]
1
2


### Đáp án Bài 2

| | c=0 | c=1 | c=2 |
|---|---|---|---|
| r=0 | 1 | 1 | 1 |
| r=1 | 1 | 2 | 1 |
| r=2 | 1 | 1 | 1 |

Đáp án `2` (tâm giữa: bốn hướng đều dài 2).

In [39]:
hand_orders_solution = [[1, 1, 1], [1, 2, 1], [1, 1, 1]]
hand_answer_solution = 2
assert orders(3, []) == hand_orders_solution
assert max(max(r) for r in hand_orders_solution) == hand_answer_solution == plus_single(3, [])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_four_solution(n: int, mines: List[List[int]]) -> int:
    grid = [[1] * n for _ in range(n)]
    for x, y in mines:
        grid[x][y] = 0
    L = [[0] * n for _ in range(n)]
    R = [[0] * n for _ in range(n)]
    U = [[0] * n for _ in range(n)]
    D = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                L[i][j] = (L[i][j - 1] if j else 0) + 1
        for j in range(n - 1, -1, -1):
            if grid[i][j]:
                R[i][j] = (R[i][j + 1] if j + 1 < n else 0) + 1
    for i in range(n):
        for j in range(n):
            if grid[i][j]:
                U[i][j] = (U[i - 1][j] if i else 0) + 1
    for i in range(n - 1, -1, -1):
        for j in range(n):
            if grid[i][j]:
                D[i][j] = (D[i + 1][j] if i + 1 < n else 0) + 1
    return max(min(L[i][j], R[i][j], U[i][j], D[i][j]) for i in range(n) for j in range(n))

assert exercise_3_four_solution(5, [[4, 2]]) == 2
assert exercise_3_four_solution(1, [[0, 0]]) == 0
assert exercise_3_four_solution(3, []) == 2
rng = random.Random(3)
for _ in range(100):
    n = rng.randint(1, 7)
    cells = [[i, j] for i in range(n) for j in range(n)]
    mines = rng.sample(cells, rng.randint(0, len(cells)))
    assert exercise_3_four_solution(n, mines) == oracle(n, mines)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_exists_solution(n: int, mines: List[List[int]], k: int) -> bool:
    banned = {(x, y) for x, y in mines}
    for r in range(k - 1, n - k + 1):
        for c in range(k - 1, n - k + 1):
            if all((r + d, c) not in banned and (r - d, c) not in banned
                   and (r, c + d) not in banned and (r, c - d) not in banned for d in range(k)):
                return True
    return False

assert exercise_4_exists_solution(5, [[4, 2]], 2) is True
assert exercise_4_exists_solution(5, [[4, 2]], 3) is False
assert exercise_4_exists_solution(1, [[0, 0]], 1) is False
rng = random.Random(4)
for _ in range(150):
    n = rng.randint(1, 7)
    cells = [[i, j] for i in range(n) for j in range(n)]
    mines = rng.sample(cells, rng.randint(0, len(cells)))
    best = oracle(n, mines)
    for k in range(1, (n + 1) // 2 + 1):
        assert exercise_4_exists_solution(n, mines, k) == (k <= best), (n, mines, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_count_centers_solution(n: int, mines: List[List[int]], k: int) -> int:
    M = four_tables(n, mines)[5]
    return sum(1 for row in M for v in row if v >= k)

def centers_oracle(n, mines, k):
    ones = {(i, j) for i in range(n) for j in range(n)} - {(x, y) for x, y in mines}
    return sum(1 for r in range(n) for c in range(n) if plus_cells(r, c, k) <= ones)

assert exercise_5_count_centers_solution(5, [], 3) == 1
assert exercise_5_count_centers_solution(5, [], 2) == 9
assert exercise_5_count_centers_solution(5, [[4, 2]], 2) == 8
assert exercise_5_count_centers_solution(3, [[1, 1]], 1) == 8
rng = random.Random(5)
for _ in range(150):
    n = rng.randint(1, 7)
    cells = [[i, j] for i in range(n) for j in range(n)]
    mines = rng.sample(cells, rng.randint(0, len(cells)))
    for k in range(1, (n + 1) // 2 + 1):
        assert exercise_5_count_centers_solution(n, mines, k) == centers_oracle(n, mines, k), (n, mines, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (một bảng `dp`, bốn bộ đếm)

```python
class Solution:
    def orderOfLargestPlusSign(self, n: int, mines: List[List[int]]) -> int:
        dp = [[n] * n for _ in range(n)]          # n = "vô cùng"; mìn = 0
        for x, y in mines:
            dp[x][y] = 0
        for i in range(n):
            left = right = up = down = 0
            for j, k in zip(range(n), reversed(range(n))):
                left  = left  + 1 if dp[i][j] else 0     # hàng i, từ trái
                right = right + 1 if dp[i][k] else 0     # hàng i, từ phải
                up    = up    + 1 if dp[j][i] else 0     # cột i, từ trên
                down  = down  + 1 if dp[k][i] else 0     # cột i, từ dưới
                dp[i][j] = min(dp[i][j], left)
                dp[i][k] = min(dp[i][k], right)
                dp[j][i] = min(dp[j][i], up)
                dp[k][i] = min(dp[k][i], down)
        return max(max(v) for v in dp)
```

Công thức nhớ nhanh: bậc tại ô `= min(trái, phải, trên, dưới)` với mỗi số là **dãy 1 liên tiếp tính cả ô**.

### Checklist trước khi submit

- [ ] Run length **tính cả ô tâm** và **đặt lại về 0** khi gặp mìn?
- [ ] Đủ **bốn** hướng và lấy `min`?
- [ ] Lưới tạo bằng list comprehension, không `[[..]*n]*n`?
- [ ] Đáp án khởi tạo từ `max` trên bảng (không ép `>= 1`) để `n = 1`, mìn `[0,0]` ra `0`?
- [ ] Chỉ số mìn `[x, y]` = (hàng, cột)?
- [ ] Đã thử: `n = 1`, lưới không mìn, mìn đúng giữa, mìn ở một cánh, đường chéo toàn mìn?
- [ ] Tên method đúng `orderOfLargestPlusSign(self, n, mines)` và trả `int`?